In [1]:
print("hi")

hi


In [2]:
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
async def root():
    return {"message": "Hello World"}

In [4]:
import random
import math
from dataclasses import dataclass
from typing import Dict, List, Optional

@dataclass
class SimConfig:
    n_cavities: int = 96
    seed: int = 42

    # Given constraints
    cycle_time_min: float = 3.8
    cycle_time_max: float = 4.5
    cavity_temp_min: float = 219.0
    cavity_temp_max: float = 221.0
    hotrunner_temp_min: float = 219.0
    hotrunner_temp_max: float = 221.0

    # Approximations (tweak if you know your process better)
    # Injection time typically sub-second to ~1.3s for fast cycles.
    injection_time_min: float = 0.55
    injection_time_max: float = 1.20

    # Small non-injection/cooling overhead: mold open/close, eject, etc.
    overhead_min: float = 0.25
    overhead_max: float = 0.55

    # Typical pressures (bar) - adjust to your plant conventions
    max_inj_pressure_min: float = 900.0
    max_inj_pressure_max: float = 1700.0

    # Cavity pressure tends to be some fraction of max injection pressure
    cavity_pressure_fraction_mean: float = 0.62
    cavity_pressure_fraction_sd: float = 0.06


class IMMProxySimulator:
    def __init__(self, cfg: SimConfig):
        self.cfg = cfg
        self.rng = random.Random(cfg.seed)

        # Per-cavity fixed offsets & scaling (simulate tooling/manifold differences)
        self.cav_temp_offset = [self.rng.uniform(-0.15, 0.15) for _ in range(cfg.n_cavities)]
        self.hr_temp_offset  = [self.rng.uniform(-0.12, 0.12) for _ in range(cfg.n_cavities)]
        self.cav_press_scale = [self.rng.uniform(0.92, 1.08) for _ in range(cfg.n_cavities)]

        # Slow drift states (simulate warming up / ambient changes)
        self.global_temp_drift = 0.0
        self.global_press_drift = 0.0
        self.t = 0  # cycle counter

    def _clamp(self, x: float, lo: float, hi: float) -> float:
        return max(lo, min(hi, x))

    def _randn(self, mu: float = 0.0, sigma: float = 1.0) -> float:
        # Box-Muller
        u1 = max(1e-12, self.rng.random())
        u2 = self.rng.random()
        z = math.sqrt(-2.0 * math.log(u1)) * math.cos(2.0 * math.pi * u2)
        return mu + sigma * z

    def next_cycle(self, anomaly_prob: float = 0.01) -> Dict[str, float]:
        cfg = self.cfg
        self.t += 1

        # --- Global slow drift (very small)
        self.global_temp_drift += self._randn(0.0, 0.003)   # ~milli-deg per cycle
        self.global_press_drift += self._randn(0.0, 0.5)    # ~0.5 bar per cycle

        # --- cycle_time (given range)
        cycle_time = self.rng.uniform(cfg.cycle_time_min, cfg.cycle_time_max)

        # --- overhead and injection_time (approx)
        overhead = self.rng.uniform(cfg.overhead_min, cfg.overhead_max)
        injection_time = self.rng.uniform(cfg.injection_time_min, cfg.injection_time_max)

        # cooling_time closes the loop with some noise, then clamp
        cooling_time = cycle_time - overhead - injection_time + self._randn(0.0, 0.05)
        # ensure cooling_time is non-negative and plausible; if not, rebalance a bit
        if cooling_time < 1.5:
            # push injection_time down a touch if cooling_time got too small
            delta = (1.5 - cooling_time)
            injection_time = self._clamp(injection_time - 0.6 * delta, cfg.injection_time_min, cfg.injection_time_max)
            cooling_time = cycle_time - overhead - injection_time + self._randn(0.0, 0.03)
        cooling_time = self._clamp(cooling_time, 1.5, 3.2)

        # --- max injection pressure (correlate with injection_time; shorter => higher)
        # Normalize injection_time to [0,1]
        it_norm = (injection_time - cfg.injection_time_min) / (cfg.injection_time_max - cfg.injection_time_min)
        base_pressure = cfg.max_inj_pressure_max - it_norm * (cfg.max_inj_pressure_max - cfg.max_inj_pressure_min)
        max_injection_pressure = base_pressure + self.global_press_drift + self._randn(0.0, 35.0)
        max_injection_pressure = self._clamp(max_injection_pressure, cfg.max_inj_pressure_min, cfg.max_inj_pressure_max)

        # --- Optional anomalies (rare spikes/drops)
        is_anomaly = (self.rng.random() < anomaly_prob)
        anomaly_factor_temp = 1.0
        anomaly_factor_press = 1.0
        if is_anomaly:
            # e.g., brief heater hiccup or pressure spike
            anomaly_factor_temp = self.rng.choice([0.7, 1.3])   # pushes some cavities toward bounds
            anomaly_factor_press = self.rng.choice([0.85, 1.15])

        # --- Per-cavity temps and pressures
        row: Dict[str, float] = {
            "cycle_time": round(cycle_time, 4),
            "injection_time": round(injection_time, 4),
            "max_injection_pressure": round(max_injection_pressure, 2),
            "cooling_time": round(cooling_time, 4),
        }

        # Pick a global setpoint center within the allowed window, with drift
        # Keep it safely inside bounds then add tiny cavity offsets/noise.
        cav_center = 220.0 + self._clamp(self.global_temp_drift, -0.25, 0.25)

        for i in range(cfg.n_cavities):
            # cavity temperature tightly controlled: offsets + small noise
            cav_temp = cav_center + self.cav_temp_offset[i] + self._randn(0.0, 0.05) * anomaly_factor_temp
            cav_temp = self._clamp(cav_temp, cfg.cavity_temp_min, cfg.cavity_temp_max)

            # hotrunner temperature: strongly correlated, slightly higher on average
            hr_temp = cav_temp + 0.05 + self.hr_temp_offset[i] + self._randn(0.0, 0.04) * anomaly_factor_temp
            hr_temp = self._clamp(hr_temp, cfg.hotrunner_temp_min, cfg.hotrunner_temp_max)

            # cavity pressure: fraction of max injection pressure + cavity scaling + noise
            frac = self._randn(cfg.cavity_pressure_fraction_mean, cfg.cavity_pressure_fraction_sd)
            frac = self._clamp(frac, 0.45, 0.85)
            cav_press = (max_injection_pressure * frac * self.cav_press_scale[i] + self._randn(0.0, 18.0)) * anomaly_factor_press
            cav_press = max(0.0, cav_press)  # pressure can't go negative

            row[f"cavity_temperature_{i+1:02d}"] = round(cav_temp, 3)
            row[f"cavity_pressure_{i+1:02d}"] = round(cav_press, 2)
            row[f"hotrunner_cav_temperature_{i+1:02d}"] = round(hr_temp, 3)

        return row

    def generate(self, n_cycles: int, anomaly_prob: float = 0.01) -> List[Dict[str, float]]:
        return [self.next_cycle(anomaly_prob=anomaly_prob) for _ in range(n_cycles)]


if __name__ == "__main__":
    sim = IMMProxySimulator(SimConfig(seed=7))
    data = sim.generate(n_cycles=5, anomaly_prob=0.02)

    # Print one row as an example
    import json
    print(json.dumps(data[0], indent=2))


{
  "cycle_time": 3.8851,
  "injection_time": 0.5972,
  "max_injection_pressure": 1648.79,
  "cooling_time": 2.981,
  "cavity_temperature_01": 219.93,
  "cavity_pressure_01": 1048.62,
  "hotrunner_cav_temperature_01": 219.969,
  "cavity_temperature_02": 219.831,
  "cavity_pressure_02": 799.31,
  "hotrunner_cav_temperature_02": 219.976,
  "cavity_temperature_03": 220.011,
  "cavity_pressure_03": 915.5,
  "hotrunner_cav_temperature_03": 220.203,
  "cavity_temperature_04": 219.826,
  "cavity_pressure_04": 1155.44,
  "hotrunner_cav_temperature_04": 219.886,
  "cavity_temperature_05": 220.037,
  "cavity_pressure_05": 1193.62,
  "hotrunner_cav_temperature_05": 220.054,
  "cavity_temperature_06": 220.048,
  "cavity_pressure_06": 1207.67,
  "hotrunner_cav_temperature_06": 220.021,
  "cavity_temperature_07": 219.977,
  "cavity_pressure_07": 1214.25,
  "hotrunner_cav_temperature_07": 219.975,
  "cavity_temperature_08": 219.972,
  "cavity_pressure_08": 1188.52,
  "hotrunner_cav_temperature_08": 2

In [5]:
import os
import time
import math
import random
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import Dict


# ---------------- Simulator (same model as before, compact) ----------------

@dataclass
class SimConfig:
    n_cavities: int = 96
    seed: int = 42

    # Given constraints
    cycle_time_min: float = 3.8
    cycle_time_max: float = 4.5
    cavity_temp_min: float = 219.0
    cavity_temp_max: float = 221.0
    hotrunner_temp_min: float = 219.0
    hotrunner_temp_max: float = 221.0

    # Approximations
    injection_time_min: float = 0.55
    injection_time_max: float = 1.20
    overhead_min: float = 0.25
    overhead_max: float = 0.55

    # Pressure (bar) — adjust if your plant uses MPa/psi
    max_inj_pressure_min: float = 900.0
    max_inj_pressure_max: float = 1700.0
    cavity_pressure_fraction_mean: float = 0.62
    cavity_pressure_fraction_sd: float = 0.06


class IMMProxySimulator:
    def __init__(self, cfg: SimConfig):
        self.cfg = cfg
        self.rng = random.Random(cfg.seed)

        self.cav_temp_offset = [self.rng.uniform(-0.15, 0.15) for _ in range(cfg.n_cavities)]
        self.hr_temp_offset  = [self.rng.uniform(-0.12, 0.12) for _ in range(cfg.n_cavities)]
        self.cav_press_scale = [self.rng.uniform(0.92, 1.08) for _ in range(cfg.n_cavities)]

        self.global_temp_drift = 0.0
        self.global_press_drift = 0.0

    def _clamp(self, x: float, lo: float, hi: float) -> float:
        return max(lo, min(hi, x))

    def _randn(self, mu: float = 0.0, sigma: float = 1.0) -> float:
        u1 = max(1e-12, self.rng.random())
        u2 = self.rng.random()
        z = math.sqrt(-2.0 * math.log(u1)) * math.cos(2.0 * math.pi * u2)
        return mu + sigma * z

    def next_cycle(self, anomaly_prob: float = 0.01) -> Dict[str, float]:
        cfg = self.cfg

        # Slow drift
        self.global_temp_drift += self._randn(0.0, 0.003)
        self.global_press_drift += self._randn(0.0, 0.5)

        # Cycle time (given)
        cycle_time = self.rng.uniform(cfg.cycle_time_min, cfg.cycle_time_max)

        overhead = self.rng.uniform(cfg.overhead_min, cfg.overhead_max)
        injection_time = self.rng.uniform(cfg.injection_time_min, cfg.injection_time_max)
        cooling_time = cycle_time - overhead - injection_time + self._randn(0.0, 0.05)

        if cooling_time < 1.5:
            delta = (1.5 - cooling_time)
            injection_time = self._clamp(injection_time - 0.6 * delta, cfg.injection_time_min, cfg.injection_time_max)
            cooling_time = cycle_time - overhead - injection_time + self._randn(0.0, 0.03)
        cooling_time = self._clamp(cooling_time, 1.5, 3.2)

        # Pressure correlated with injection_time (shorter => higher)
        it_norm = (injection_time - cfg.injection_time_min) / (cfg.injection_time_max - cfg.injection_time_min)
        base_pressure = cfg.max_inj_pressure_max - it_norm * (cfg.max_inj_pressure_max - cfg.max_inj_pressure_min)
        max_injection_pressure = self._clamp(
            base_pressure + self.global_press_drift + self._randn(0.0, 35.0),
            cfg.max_inj_pressure_min,
            cfg.max_inj_pressure_max,
        )

        # Rare anomalies
        anomaly = (self.rng.random() < anomaly_prob)
        anomaly_factor_temp = self.rng.choice([0.7, 1.3]) if anomaly else 1.0
        anomaly_factor_press = self.rng.choice([0.85, 1.15]) if anomaly else 1.0

        row: Dict[str, float] = {
            "cycle_time": round(cycle_time, 4),
            "injection_time": round(injection_time, 4),
            "max_injection_pressure": round(max_injection_pressure, 2),
            "cooling_time": round(cooling_time, 4),
        }

        cav_center = 220.0 + self._clamp(self.global_temp_drift, -0.25, 0.25)

        for i in range(cfg.n_cavities):
            cav_temp = cav_center + self.cav_temp_offset[i] + self._randn(0.0, 0.05) * anomaly_factor_temp
            cav_temp = self._clamp(cav_temp, cfg.cavity_temp_min, cfg.cavity_temp_max)

            hr_temp = cav_temp + 0.05 + self.hr_temp_offset[i] + self._randn(0.0, 0.04) * anomaly_factor_temp
            hr_temp = self._clamp(hr_temp, cfg.hotrunner_temp_min, cfg.hotrunner_temp_max)

            frac = self._clamp(self._randn(cfg.cavity_pressure_fraction_mean, cfg.cavity_pressure_fraction_sd), 0.45, 0.85)
            cav_press = (max_injection_pressure * frac * self.cav_press_scale[i] + self._randn(0.0, 18.0)) * anomaly_factor_press
            cav_press = max(0.0, cav_press)

            row[f"cavity_temperature_{i+1:02d}"] = round(cav_temp, 3)
            row[f"cavity_pressure_{i+1:02d}"] = round(cav_press, 2)
            row[f"hotrunner_cav_temperature_{i+1:02d}"] = round(hr_temp, 3)

        return row


# ---------------- Writer: one file per variable ----------------

def safe_filename(var_name: str) -> str:
    # keep alnum, dash, underscore, dot; replace others with underscore
    out = []
    for ch in var_name:
        if ch.isalnum() or ch in ("-", "_", "."):
            out.append(ch)
        else:
            out.append("_")
    return "".join(out)

def append_row(path: str, timestamp: str, value) -> None:
    new_file = not os.path.exists(path)
    with open(path, "a", encoding="utf-8") as f:
        if new_file:
            f.write("timestamp,value\n")
        f.write(f"{timestamp},{value}\n")


def run_for_one_minute(
    data_dir: str = "data",
    seed: int = 7,
    anomaly_prob: float = 0.01,
    use_utc: bool = True,
) -> None:
    os.makedirs(data_dir, exist_ok=True)

    sim = IMMProxySimulator(SimConfig(seed=seed))

    end_time = time.time() + 60.0

    while time.time() < end_time:
        # Generate one cycle worth of data
        row = sim.next_cycle(anomaly_prob=anomaly_prob)

        # Timestamp at generation time (ISO8601 with milliseconds)
        now = datetime.now(timezone.utc) if use_utc else datetime.now().astimezone()
        ts = now.isoformat(timespec="milliseconds")

        # Append each variable into its own file
        for var, val in row.items():
            filename = safe_filename(var) + ".csv"
            path = os.path.join(data_dir, filename)
            append_row(path, ts, val)

        # Sleep approximately the cycle time (so the cadence matches your simulated cycle)
        # If you want fixed-rate sampling instead, replace with time.sleep(0.1) or similar.
        time.sleep(float(row["cycle_time"]))


if __name__ == "__main__":
    run_for_one_minute(
        data_dir="data",
        seed=7,
        anomaly_prob=0.02,
        use_utc=True,
    )
    print("Done. Files written to ./data/")


Done. Files written to ./data/


In [ ]:
from datetime import datetime
from pydantic import BaseModel

class Delivery(BaseModel):
    timestamp: datetime
    dimensions: tuple[int, int]


PydanticImportError: `BaseSettings` has been moved to the `pydantic-settings` package. See https://docs.pydantic.dev/2.12/migration/#basesettings-has-moved-to-pydantic-settings for more details.

For further information visit https://errors.pydantic.dev/2.12/u/import-error